# 02 · Preprocessing and leakage-safe splits

This notebook calls the existing metadata importer and `covid_xray.data` package. It does not reimplement preprocessing. Obtain the exact licensed COVIDx CXR-3A version and pinned label files by following `docs/DATASET.md`; never commit the generated CSV files or medical images.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

RAW = ROOT / 'data' / 'raw' / 'covidx'
TRAIN_LABELS = ROOT / 'data' / 'raw' / 'train_COVIDx_CXR-3A.txt'
TEST_LABELS = ROOT / 'data' / 'raw' / 'test_COVIDx_CXR-3A.txt'
PATIENTS = ROOT / 'data' / 'processed' / 'patients.csv'
INDEX = ROOT / 'data' / 'processed' / 'index.csv'
MANIFEST = ROOT / 'data' / 'processed' / 'manifest.csv'

In [ ]:
required = [RAW, TRAIN_LABELS, TEST_LABELS]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Install the licensed dataset first; missing: ' + ', '.join(missing))
PATIENTS.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
if not PATIENTS.exists():
    subprocess.run([sys.executable, 'scripts/import_covidx_metadata.py', '--labels', str(TRAIN_LABELS), str(TEST_LABELS), '--data-dir', str(RAW), '--output', str(PATIENTS)], check=True)
subprocess.run([sys.executable, '-m', 'covid_xray.data', 'index', '--data-dir', str(RAW), '--patients', str(PATIENTS), '--output', str(INDEX)], check=True)
subprocess.run([sys.executable, '-m', 'covid_xray.data', 'split', '--index', str(INDEX), '--output', str(MANIFEST), '--val-size', '0.15'], check=True)

In [ ]:
from covid_xray.data import load_manifest

frame = load_manifest(MANIFEST)
display(frame.groupby(['split', 'label']).size().rename('images').unstack(fill_value=0))
assert frame.groupby('patient_id').split.nunique().max() == 1
assert frame.groupby('sha256').split.nunique().max() == 1
assert frame.groupby('perceptual_hash').split.nunique().max() == 1
print(f'Validated manifest: {MANIFEST}')